In [ ]:
import arcpy
import os

# =========================
# 1) Environment and input
# =========================
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

china_province = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\china_pro2.shp"

out_grid_points_wgs84 = os.path.join(scratch_gdb, "China_20km_grid_points_WGS84")

cell_size = 20000  # 20 km (meters)

print("Read provincial boundary data...")
desc = arcpy.Describe(china_province)
sr_in = desc.spatialReference

# ✅ Your provincial boundary field: ADCODE99 + NAME
need_fields = ["ADCODE99", "NAME"]
have_fields = [f.name for f in arcpy.ListFields(china_province)]
missing = [f for f in need_fields if f not in have_fields]
if missing:
    raise RuntimeError(f":{missing}. please check{china_province}.")

# =========================
# 2) Spatial-reference helpers
# =========================
def is_degree_sr(spref: arcpy.SpatialReference) -> bool:
    try:
        return (spref.type == "Geographic") or (spref.linearUnitName is None) or ("Degree" in (spref.linearUnitName or ""))
    except:
        return True

# Recommended projection: Asia North Albers Equal Area Conic (unit: meters)
try:
    sr_grid = arcpy.SpatialReference("Asia North Albers Equal Area Conic")
except:
    sr_grid = arcpy.SpatialReference(3857)  # Full details (unit: meters)

china_for_grid = china_province
if is_degree_sr(sr_in):
    print("Detected provincial boundaries in geographic coordinate system (degrees). Projecting to meter unit projected coordinate system to generate 20km grid...")
    china_for_grid = os.path.join(scratch_gdb, "china_province_projected_for_grid")
    arcpy.management.Project(china_province, china_for_grid, sr_grid)
else:
    sr_grid = sr_in

china_extent = arcpy.Describe(china_for_grid).extent
arcpy.env.outputCoordinateSystem = sr_grid

# =========================
# 3) 20km(POLYGON)
# =========================
print("Create a 20km fishing net (POLYGON)...")
fishnet = os.path.join(scratch_gdb, "fishnet_20km_poly")

origin_coord = f"{china_extent.XMin} {china_extent.YMin}"
y_axis_coord = f"{china_extent.XMin} {china_extent.YMax}"
corner_coord = f"{china_extent.XMax} {china_extent.YMax}"

arcpy.management.CreateFishnet(
    out_feature_class=fishnet,
    origin_coord=origin_coord,
    y_axis_coord=y_axis_coord,
    cell_width=cell_size,
    cell_height=cell_size,
    number_rows=0,
    number_columns=0,
    corner_coord=corner_coord,
    labels="NO_LABELS",
    template="#",
    geometry_type="POLYGON"
)

# =========================
# 4) Fishing net rotation center point (center of mass)
# =========================
print("Generate the center point (center of mass) of the fishing net...")
grid_points = os.path.join(scratch_gdb, "fishnet_20km_centroids")
arcpy.management.FeatureToPoint(fishnet, grid_points, "CENTROID")

# =========================
# 5) Crop to China
# =========================
print("Cropped to China...")
clipped_points = os.path.join(scratch_gdb, "China_20km_grid_points_projected")
arcpy.analysis.Clip(grid_points, china_for_grid, clipped_points)

# =========================
# 6) Spatial Join: bring out ADCODE99 + NAME
# =========================
print("Add province attributes (ADCODE99/NAME) to grid points...")
points_joined = os.path.join(scratch_gdb, "China_20km_grid_points_joined")

arcpy.analysis.SpatialJoin(
    target_features=clipped_points,
    join_features=china_for_grid,
    out_feature_class=points_joined,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

# =========================
# 7) Project to WGS84
# =========================
print("Project to WGS84...")
wgs84 = arcpy.SpatialReference(4326)
arcpy.management.Project(points_joined, out_grid_points_wgs84, wgs84)

# =========================
# 8) Add Province field and fill it with NAME
# =========================
print("Province NAME ...")
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
if "Province" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Province", "TEXT", field_length=100)

# Calculate Province = NAME
# Note: If NAME becomes NAME_1 after join, compatibility will be handled here
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
name_field = "NAME" if "NAME" in fields_out else ("NAME_1" if "NAME_1" in fields_out else None)
if name_field is None:
    raise RuntimeError("The NAME or NAME_1 field was not found in the output results, please check the Spatial Join output fields.")

with arcpy.da.UpdateCursor(out_grid_points_wgs84, [name_field, "Province"]) as cursor:
    for n, p in cursor:
        cursor.updateRow((n, n))

# =========================
# 9)
# =========================
print("Add latitude and longitude fields...")
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
if "Longitude" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Longitude", "DOUBLE")
if "Latitude" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Latitude", "DOUBLE")

with arcpy.da.UpdateCursor(out_grid_points_wgs84, ["SHAPE@X", "SHAPE@Y", "Longitude", "Latitude"]) as cursor:
    for x, y, lon, lat in cursor:
        cursor.updateRow((x, y, x, y))

# =========================
# 10) ()
# =========================
print("Add results to the current map...")
project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap
active_map.addDataFromPath(out_grid_points_wgs84)

count = int(arcpy.management.GetCount(out_grid_points_wgs84)[0])
print(f"Done! Generated {count} 20-km uniform grid points within China.")
print(":Longitude, Latitude, ADCODE99, Province(NAME)")
print(f"Output elements:{out_grid_points_wgs84}")


In [ ]:
import arcpy
import os

# =========================
# 1) Environment and input
# =========================
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

china_province = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\china_pro2.shp"

out_grid_points_wgs84 = os.path.join(scratch_gdb, "China_10km_grid_points_WGS84")

cell_size = 10000  # 10 km (meters)

print("Read provincial boundary data...")
desc = arcpy.Describe(china_province)
sr_in = desc.spatialReference

# ✅ Your provincial boundary field: ADCODE99 + NAME
need_fields = ["ADCODE99", "NAME"]
have_fields = [f.name for f in arcpy.ListFields(china_province)]
missing = [f for f in need_fields if f not in have_fields]
if missing:
    raise RuntimeError(f":{missing}. please check{china_province}.")

# =========================
# 2) Spatial-reference helpers
# =========================
def is_degree_sr(spref: arcpy.SpatialReference) -> bool:
    try:
        return (spref.type == "Geographic") or (spref.linearUnitName is None) or ("Degree" in (spref.linearUnitName or ""))
    except:
        return True

# Recommended projection: Asia North Albers Equal Area Conic (unit: meters)
try:
    sr_grid = arcpy.SpatialReference("Asia North Albers Equal Area Conic")
except:
    sr_grid = arcpy.SpatialReference(3857)  # Full details (unit: meters)

china_for_grid = china_province
if is_degree_sr(sr_in):
    print("().10km...")
    china_for_grid = os.path.join(scratch_gdb, "china_province_projected_for_grid")
    arcpy.management.Project(china_province, china_for_grid, sr_grid)
else:
    sr_grid = sr_in

china_extent = arcpy.Describe(china_for_grid).extent
arcpy.env.outputCoordinateSystem = sr_grid

# =========================
# 3) 10km(POLYGON)
# =========================
print("10km(POLYGON)...")
fishnet = os.path.join(scratch_gdb, "fishnet_10km_poly")

origin_coord = f"{china_extent.XMin} {china_extent.YMin}"
y_axis_coord = f"{china_extent.XMin} {china_extent.YMax}"
corner_coord = f"{china_extent.XMax} {china_extent.YMax}"

arcpy.management.CreateFishnet(
    out_feature_class=fishnet,
    origin_coord=origin_coord,
    y_axis_coord=y_axis_coord,
    cell_width=cell_size,
    cell_height=cell_size,
    number_rows=0,
    number_columns=0,
    corner_coord=corner_coord,
    labels="NO_LABELS",
    template="#",
    geometry_type="POLYGON"
)

# =========================
# 4) Fishing net rotation center point (center of mass)
# =========================
print("Generate the center point (center of mass) of the fishing net...")
grid_points = os.path.join(scratch_gdb, "fishnet_10km_centroids")
arcpy.management.FeatureToPoint(fishnet, grid_points, "CENTROID")

# =========================
# 5) Crop to China
# =========================
print("Cropped to China...")
clipped_points = os.path.join(scratch_gdb, "China_10km_grid_points_projected")
arcpy.analysis.Clip(grid_points, china_for_grid, clipped_points)

# =========================
# 6) Spatial Join: bring out ADCODE99 + NAME
# =========================
print("Add province attributes (ADCODE99/NAME) to grid points...")
points_joined = os.path.join(scratch_gdb, "China_10km_grid_points_joined")

arcpy.analysis.SpatialJoin(
    target_features=clipped_points,
    join_features=china_for_grid,
    out_feature_class=points_joined,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

# =========================
# 7) Project to WGS84
# =========================
print("Project to WGS84...")
wgs84 = arcpy.SpatialReference(4326)
arcpy.management.Project(points_joined, out_grid_points_wgs84, wgs84)

# =========================
# 8) Add Province field and fill it with NAME
# =========================
print("Province NAME ...")
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
if "Province" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Province", "TEXT", field_length=100)

# Calculate Province = NAME
# Note: If NAME becomes NAME_1 after join, compatibility will be handled here
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
name_field = "NAME" if "NAME" in fields_out else ("NAME_1" if "NAME_1" in fields_out else None)
if name_field is None:
    raise RuntimeError("The NAME or NAME_1 field was not found in the output results, please check the Spatial Join output fields.")

with arcpy.da.UpdateCursor(out_grid_points_wgs84, [name_field, "Province"]) as cursor:
    for n, p in cursor:
        cursor.updateRow((n, n))

# =========================
# 9)
# =========================
print("Add latitude and longitude fields...")
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
if "Longitude" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Longitude", "DOUBLE")
if "Latitude" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Latitude", "DOUBLE")

with arcpy.da.UpdateCursor(out_grid_points_wgs84, ["SHAPE@X", "SHAPE@Y", "Longitude", "Latitude"]) as cursor:
    for x, y, lon, lat in cursor:
        cursor.updateRow((x, y, x, y))

# =========================
# 10) ()
# =========================
print("Add results to the current map...")
project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap
active_map.addDataFromPath(out_grid_points_wgs84)

count = int(arcpy.management.GetCount(out_grid_points_wgs84)[0])
print(f"Done! Generated {count} 10-km uniform grid points within China.")
print(":Longitude, Latitude, ADCODE99, Province(NAME)")
print(f"Output elements:{out_grid_points_wgs84}")
